# 🩺🤖 Assistant Pédagogique — TSAPBENG 2027 (version publiable)

**Version nettoyée pour GitHub : aucune vraie clé n'est incluse.**

⚠️ Avant d'exécuter : remplacez `colle_ta_cle_groq_ici` et `colle_ta_cle_tavily_ici` par VOS clés,
**localement seulement**. Ne recommitez jamais le fichier avec de vraies clés dedans.


# 🩺🤖 Assistant Pédagogique.

**Écoles de santé du Cameroun** · mise à jour 2026

Version nettoyée et cohérente : un **socle unique** (enrichi : sources multi-domaines, CSV/Excel, mémoire d'index), puis vos **générateurs en classes** (cours, activités, évaluation) et les **supports**.

⚠️ **Sécurité** : ne mettez jamais de vraie clé en clair si vous partagez ce fichier. Ici, les clés sont des **espaces réservés** à remplacer.

## 1) Installation (une seule fois)

In [ ]:
%pip install -q -U langchain langchain-community langchain-groq langchain-huggingface tavily-python faiss-cpu sentence-transformers pypdf docx2txt python-docx python-pptx pandas openpyxl
print("✅ Installation terminée.")

## 2) Vos clés + la fonction `demander`

⚠️ Remplacez les espaces réservés par vos clés. **Ne partagez pas** le fichier une fois vos vraies clés collées.

In [ ]:
import os, docx
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage

os.environ["GROQ_API_KEY"]   = "colle_ta_cle_groq_ici"     # console.groq.com
os.environ["TAVILY_API_KEY"] = "colle_ta_cle_tavily_ici"   # app.tavily.com

llm = ChatGroq(model="llama-3.1-8b-instant")

def demander(systeme, question):
    return llm.invoke([SystemMessage(content=systeme), HumanMessage(content=question)]).content

def exporter_texte_word(titre, contenu, fichier):
    doc = docx.Document(); doc.add_heading(titre, 0)
    for para in contenu.split("\n"):
        doc.add_paragraph(para)
    doc.save(fichier); print("📄", fichier)

print("✅ Prêt.")

## 3) La bibliothèque — enrichie (multi-domaines, CSV/Excel, mémoire d'index)

Nouveautés intégrées depuis votre travail : le web est filtré **par domaine** (santé / biostat / recherche), on lit aussi les **CSV et Excel**, et l'index est **sauvegardé sur disque** (`faiss_index`) pour ne pas tout réindexer à chaque fois. Mettez `RECONSTRUIRE = True` après avoir ajouté des documents.

In [ ]:
import glob
import pandas as pd
from tavily import TavilyClient
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader, TextLoader, CSVLoader

DOSSIER = "ma_bibliotheque"; INDEX_DIR = "faiss_index"
os.makedirs(DOSSIER, exist_ok=True)
tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

# Sites fiables par domaine : le web est filtré selon la question
SOURCES = {
    "sante":     ["who.int","afro.who.int","cdc.gov","unicef.org","minsante.cm",
                  "pubmed.ncbi.nlm.nih.gov","ncbi.nlm.nih.gov","nih.gov","sciencedirect.com"],
    "biostat":   ["cran.r-project.org","r-project.org","jamovi.org","jasp-stats.org"],
    "recherche": ["equator-network.org","prisma-statement.org","consort-statement.org",
                  "strobe-statement.org","cairn.info","hal.science"],
}
def choisir_domaines(q):
    q = q.lower()

    if any(x in q for x in [
        "biostat",
        "anova",
        "régression",
        "regression",
        "spss",
        "jamovi",
        "statistique",
        "statistiques"
    ]):
        return SOURCES["biostat"]

    if any(x in q for x in [
        "épidémiologie",
        "epidemiologie",
        "incidence",
        "prévalence",
        "prevalence",
        "risque relatif",
        "odds ratio",
        "cohorte",
        "cas-témoins"
    ]):
        return SOURCES["epidemiologie"]

    if any(x in q for x in [
        "méthode",
        "méthodologie",
        "methodologie",
        "prisma",
        "consort",
        "strobe",
        "revue systématique",
        "revue de littérature",
        "article scientifique"
    ]):
        return SOURCES["recherche"]

    if any(x in q for x in [
        "nutrition",
        "malnutrition",
        "alimentation",
        "anémie",
        "carence"
    ]):
        return SOURCES["nutrition"]

    if any(x in q for x in [
        "grossesse",
        "maternel",
        "maternité",
        "enfant",
        "infantile",
        "nouveau-né",
        "vaccination"
    ]):
        return SOURCES["maternel_enfant"]

    return SOURCES["sante"]

def charger(dossier):
    docs = []
    for f in glob.glob(os.path.join(dossier, "**", "*"), recursive=True):
        if os.path.isdir(f): continue
        ext = f.lower().rsplit(".", 1)[-1]
        try:
            if   ext == "pdf":            docs += PyPDFLoader(f).load()
            elif ext in ("docx","doc"):   docs += Docx2txtLoader(f).load()
            elif ext in ("txt","md"):     docs += TextLoader(f, encoding="utf-8").load()
            elif ext == "csv":            docs += CSVLoader(f).load()
            elif ext in ("xlsx","xls"):   docs.append(Document(page_content=pd.read_excel(f).to_string(),
                                                               metadata={"source": f}))
        except Exception as e:
            print("  ✗", os.path.basename(f), e)
    return docs

documents = charger(DOSSIER)
if not documents:
    open(os.path.join(DOSSIER,"exemple.txt"),"w",encoding="utf-8").write(
        "Le paludisme se transmet par le moustique anophèle. Symptômes : fièvre, frissons. Prévention : moustiquaire imprégnée.")
    documents = charger(DOSSIER)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

RECONSTRUIRE = False   # -> True quand vous ajoutez de nouveaux documents
if os.path.exists(INDEX_DIR) and not RECONSTRUIRE:
    index = FAISS.load_local(INDEX_DIR, embeddings, allow_dangerous_deserialization=True)
else:
    chunks = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150).split_documents(documents)
    index = FAISS.from_documents(chunks, embeddings)
    index.save_local(INDEX_DIR)

chercheur = index.as_retriever(search_kwargs={"k": 3})

def chercher_contexte(sujet):
    docs = chercheur.invoke(sujet)
    contexte = "\n\n".join(d.page_content for d in docs)
    if contexte.strip():
        return contexte, [os.path.basename(str(d.metadata.get("source","local"))) for d in docs], "📁 local"
    res = tavily.search(query=sujet, include_domains=choisir_domaines(sujet), max_results=5)["results"]
    return "\n\n".join(r["content"] for r in res), [r["url"] for r in res], "🌐 en ligne"

def _systeme(sujet):
    contexte, _, _ = chercher_contexte(sujet)
    return f"Tu es un enseignant de santé au Cameroun. Réponds en français, à partir du contexte.\n\nContexte :\n{contexte}"

def ask(question):
    contexte, sources, origine = chercher_contexte(question)
    systeme = ("Tu es un professeur de santé. Réponds en français en utilisant UNIQUEMENT le contexte, "
               "et cite tes sources.\n\nContexte :\n" + contexte)
    return demander(systeme, question), sources, origine

print(f"✅ Bibliothèque prête ({len(documents)} documents).")

### Essai rapide de l'assistant (sans boucle bloquante)

Remplace l'ancienne boucle `while True: input()` (qui bloquait le notebook) par un simple appel.

In [ ]:
import os
import glob
import json
import hashlib
import pandas as pd

from tavily import TavilyClient
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import (
    PyPDFLoader,
    Docx2txtLoader,
    TextLoader,
    CSVLoader
)


# ============================================================
# 1. CONFIGURATION
# ============================================================

DOSSIER = "ma_bibliotheque"
INDEX_DIR = "faiss_index"
MANIFEST_FILE = "bibliotheque_manifest.json"

os.makedirs(DOSSIER, exist_ok=True)

# Clé Tavily
tavily = TavilyClient(
    api_key=os.environ["TAVILY_API_KEY"]
)
# ============================================================
# 2. sources_web
# ============================================================
def charger_sources_web(fichier="sources_web.json"):
    with open(fichier, "r", encoding="utf-8") as f:
        return json.load(f)

SOURCES = charger_sources_web()
print("🌐 Sources Web chargées :")
for categorie, sites in SOURCES.items():
    print(f"  {categorie} : {len(sites)} sites")
# ============================================================
# 3. CALCUL DE L'EMPREINTE D'UN FICHIER
# ============================================================

def calculer_hash(fichier):
    """
    Calcule une empreinte SHA-256 du fichier.

    Si le fichier est modifié, son hash change.
    Cela permet de détecter automatiquement les modifications.
    """

    sha256 = hashlib.sha256()

    with open(fichier, "rb") as f:
        for bloc in iter(lambda: f.read(1024 * 1024), b""):
            sha256.update(bloc)

    return sha256.hexdigest()


# ============================================================
# 4. CHARGEMENT DU MANIFEST
# ============================================================

def charger_manifest():
    """
    Charge le registre des fichiers déjà intégrés.
    """

    if not os.path.exists(MANIFEST_FILE):
        return {}

    try:
        with open(
            MANIFEST_FILE,
            "r",
            encoding="utf-8"
        ) as f:
            return json.load(f)

    except Exception as e:
        print("⚠️ Impossible de lire le manifest :", e)
        return {}


# ============================================================
# 5. SAUVEGARDE DU MANIFEST
# ============================================================

def sauvegarder_manifest(manifest):
    """
    Sauvegarde le registre des fichiers.
    """

    with open(
        MANIFEST_FILE,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            manifest,
            f,
            ensure_ascii=False,
            indent=2
        )


# ============================================================
# 6. DÉTECTION DES DOCUMENTS
# ============================================================

def trouver_fichiers():
    """
    Recherche tous les documents compatibles
    dans ma_bibliotheque et ses sous-dossiers.
    """

    extensions = {
        ".pdf",
        ".docx",
        ".doc",
        ".txt",
        ".md",
        ".csv",
        ".xlsx",
        ".xls"
    }

    fichiers = []

    for f in glob.glob(
        os.path.join(DOSSIER, "**", "*"),
        recursive=True
    ):

        if not os.path.isfile(f):
            continue

        extension = os.path.splitext(f)[1].lower()

        if extension in extensions:
            fichiers.append(f)

    return sorted(fichiers)


# ============================================================
# 7. CHARGEMENT D'UN DOCUMENT
# ============================================================

def charger_un_document(fichier):
    """
    Charge un seul fichier selon son extension.
    """

    extension = os.path.splitext(fichier)[1].lower()

    try:

        if extension == ".pdf":

            return PyPDFLoader(fichier).load()

        elif extension in [".docx", ".doc"]:

            return Docx2txtLoader(fichier).load()

        elif extension in [".txt", ".md"]:

            return TextLoader(
                fichier,
                encoding="utf-8"
            ).load()

        elif extension == ".csv":

            return CSVLoader(fichier).load()

        elif extension in [".xlsx", ".xls"]:

            df = pd.read_excel(fichier)

            texte = df.to_string(index=False)

            return [
                Document(
                    page_content=texte,
                    metadata={
                        "source": fichier
                    }
                )
            ]

        else:

            print(
                "⚠️ Format non supporté :",
                fichier
            )

            return []

    except Exception as e:

        print(
            "✗ Erreur avec",
            os.path.basename(fichier),
            ":",
            e
        )

        return []


# ============================================================
# 8. EMBEDDINGS
# ============================================================

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)


# ============================================================
# 9. CHARGE OU CRÉE L'INDEX FAISS
# ============================================================

if os.path.exists(INDEX_DIR):

    print("📚 Chargement de l'index FAISS existant...")

    index = FAISS.load_local(
        INDEX_DIR,
        embeddings,
        allow_dangerous_deserialization=True
    )

    print("✅ Index existant chargé.")

else:

    print("📚 Aucun index trouvé.")

    index = None


# ============================================================
# 10. SYNCHRONISATION AUTOMATIQUE
# ============================================================

def synchroniser_bibliotheque():
    """
    Détecte les nouveaux fichiers et les fichiers modifiés.

    Les nouveaux documents sont ajoutés à FAISS.

    Les documents modifiés sont également ajoutés
    avec leur nouvelle version.

    L'ancien index n'est jamais supprimé.
    """

    global index

    fichiers = trouver_fichiers()

    manifest = charger_manifest()

    if not fichiers:

        print("📁 Aucun document trouvé.")

        return

    nouveaux = []
    modifies = []
    inchanges = []

    print()
    print("🔎 Analyse de la bibliothèque...")
    print()

    # --------------------------------------------------------
    # Détection
    # --------------------------------------------------------

    for fichier in fichiers:

        chemin_absolu = os.path.abspath(fichier)

        hash_actuel = calculer_hash(fichier)

        ancien_hash = manifest.get(
            chemin_absolu,
            {}
        ).get("hash")

        if ancien_hash is None:

            nouveaux.append(
                (fichier, hash_actuel)
            )

        elif ancien_hash != hash_actuel:

            modifies.append(
                (fichier, hash_actuel)
            )

        else:

            inchanges.append(fichier)

    # --------------------------------------------------------
    # Affichage
    # --------------------------------------------------------

    print(
        f"📄 Documents trouvés : {len(fichiers)}"
    )

    print(
        f"🆕 Nouveaux : {len(nouveaux)}"
    )

    print(
        f"🔄 Modifiés : {len(modifies)}"
    )

    print(
        f"✓ Inchangés : {len(inchanges)}"
    )

    # --------------------------------------------------------
    # Documents à intégrer
    # --------------------------------------------------------

    a_ajouter = nouveaux + modifies

    if not a_ajouter:

        print()
        print(
            "✅ Bibliothèque déjà à jour."
        )

        return

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=150
    )

    tous_les_chunks = []

    # --------------------------------------------------------
    # Chargement
    # --------------------------------------------------------

    for fichier, hash_fichier in a_ajouter:

        print()
        print(
            "📥 Traitement :",
            os.path.relpath(fichier, DOSSIER)
        )

        documents = charger_un_document(fichier)

        if not documents:

            print(
                "⚠️ Aucun contenu récupéré."
            )

            continue

        # Ajouter des métadonnées utiles
        for document in documents:

            document.metadata["source"] = fichier

            document.metadata["nom_fichier"] = (
                os.path.basename(fichier)
            )

            document.metadata["chemin"] = (
                os.path.abspath(fichier)
            )

            document.metadata["hash"] = hash_fichier

        chunks = splitter.split_documents(
            documents
        )

        print(
            f"   → {len(documents)} document(s)"
        )

        print(
            f"   → {len(chunks)} fragment(s)"
        )

        tous_les_chunks.extend(chunks)

        # Mise à jour du manifest
        manifest[os.path.abspath(fichier)] = {
            "hash": hash_fichier,
            "nom": os.path.basename(fichier),
            "type": os.path.splitext(fichier)[1].lower()
        }

    # --------------------------------------------------------
    # Ajout à FAISS
    # --------------------------------------------------------

    if tous_les_chunks:

        print()
        print(
            "🧠 Création des embeddings..."
        )

        if index is None:

            print(
                "🆕 Création du premier index FAISS..."
            )

            index = FAISS.from_documents(
                tous_les_chunks,
                embeddings
            )

        else:

            print(
                "➕ Ajout au FAISS existant..."
            )

            index.add_documents(
                tous_les_chunks
            )

        # Sauvegarde
        index.save_local(
            INDEX_DIR
        )

        sauvegarder_manifest(
            manifest
        )

        print()
        print(
            "✅ Bibliothèque mise à jour."
        )

        print(
            f"   {len(tous_les_chunks)} nouveaux fragments ajoutés."
        )

    else:

        print()
        print(
            "⚠️ Aucun nouveau contenu à ajouter."
        )


# ============================================================
# 11. SYNCHRONISATION AU DÉMARRAGE
# ============================================================

synchroniser_bibliotheque()


# ============================================================
# 12. RETRIEVER
# ============================================================

chercheur = index.as_retriever(
    search_kwargs={
        "k": 3
    }
)


# ============================================================
# 13. RECHERCHE DANS LA BIBLIOTHÈQUE
# ============================================================

def chercher_contexte_local(sujet):

    docs = chercheur.invoke(sujet)

    if not docs:

        return "", []

    contexte = "\n\n".join(
        d.page_content
        for d in docs
    )

    sources = [
        os.path.basename(
            str(
                d.metadata.get(
                    "source",
                    "local"
                )
            )
        )
        for d in docs
    ]

    return contexte, sources


# ============================================================
# 14. RECHERCHE WEB
# ============================================================

def chercher_web(sujet):

    domaines = choisir_domaines(
        sujet
    )

    try:

        resultats = tavily.search(
            query=sujet,
            include_domains=domaines,
            max_results=5
        )

        contenu = "\n\n".join(
            r.get("content", "")
            for r in resultats["results"]
        )

        sources = [
            r.get("url", "")
            for r in resultats["results"]
        ]

        return contenu, sources

    except Exception as e:

        print(
            "⚠️ Erreur recherche web :",
            e
        )

        return "", []


# ============================================================
# 15. RECHERCHE INTELLIGENTE
# ============================================================

def chercher_contexte(sujet):

    contexte_local, sources_locales = (
        chercher_contexte_local(sujet)
    )

    # Si on trouve quelque chose dans la bibliothèque
    if contexte_local.strip():

        return (
            contexte_local,
            sources_locales,
            "📁 local"
        )

    # Sinon recherche web
    contexte_web, sources_web = (
        chercher_web(sujet)
    )

    return (
        contexte_web,
        sources_web,
        "🌐 en ligne"
    )


# ============================================================
# 16. SYSTÈME
# ============================================================

def _systeme(sujet):

    contexte, _, _ = chercher_contexte(
        sujet
    )

    return f"""
Tu es un enseignant de santé au Cameroun.

Réponds en français.

Utilise uniquement les informations présentes
dans le contexte fourni.

Ne fabrique pas de références.

Si le contexte ne permet pas de répondre correctement,
indique clairement que l'information disponible
est insuffisante.

Contexte :

{contexte}
"""


# ============================================================
# 17. QUESTION À L'IA
# ============================================================

def ask(question):

    contexte, sources, origine = (
        chercher_contexte(question)
    )

    systeme = f"""
Tu es un professeur de santé.

Réponds en français.

Utilise UNIQUEMENT le contexte fourni.

Cite les sources utilisées.

Si les informations sont insuffisantes,
dis-le clairement au lieu d'inventer.

Contexte :

{contexte}
"""

    # IMPORTANT :
    # La fonction "demander()" doit déjà exister
    # dans ton projet.

    reponse = demander(
        systeme,
        question
    )

    return (
        reponse,
        sources,
        origine
    )


# ============================================================
# 18. MESSAGE FINAL
# ============================================================

nombre_documents = len(
    trouver_fichiers()
)

print()
print("=" * 60)
print(
    f"✅ Bibliothèque prête : {nombre_documents} document(s)"
)
print("=" * 60)

In [ ]:
reponse, sources, origine = ask("Quelles sont les mesures de prévention du paludisme au Cameroun ?")
print(origine, "—", sources)
print(reponse)

## 3 bis) 🛡️ Fiabilité — des réponses vérifiables (anti-hallucination)

En santé, une **hallucination** (affirmation fausse dite avec assurance) est dangereuse. Cette brique rend chaque réponse **traçable et prudente** :
1. 🔢 **citations numérotées** `[1]`, `[2]` reliées aux sources ;
2. 📊 **score de confiance** calculé à partir de la pertinence des passages (< 60 % = « à vérifier ») ;
3. 🚫 **signalement du hors-corpus** (l'IA dit « non trouvé » au lieu d'inventer) ;
4. 🔎 **vérificateur** qui repère les affirmations non justifiées.

> C'est **le même moteur** que celui de l'application `app.py` : mêmes fonctions, même seuil.

In [ ]:
SEUIL_PERTINENCE = 0.30   # en dessous, l'info est considérée « hors-corpus »

def _src(d):
    return os.path.basename(str(d.metadata.get("source", "local")))

def recuperer_avec_scores(question, k=4):
    """Renvoie (texte, source, pertinence 0..1). Plus c'est haut, mieux c'est."""
    try:
        res = index.similarity_search_with_relevance_scores(question, k=k)
        return [(d.page_content, _src(d), max(0.0, min(1.0, float(s)))) for d, s in res]
    except Exception:
        res = index.similarity_search_with_score(question, k=k)   # repli : distance -> pertinence
        return [(d.page_content, _src(d), 1.0 / (1.0 + float(dist))) for d, dist in res]

def _contexte_numerote(question, k=4):
    """Prépare le contexte numéroté + les sources + la confiance (moteur commun)."""
    passages = recuperer_avec_scores(question, k)
    pertinents = [p for p in passages if p[2] >= SEUIL_PERTINENCE]
    if pertinents:
        contexte, sources = "", []
        for i, (texte, src, sc) in enumerate(pertinents, 1):
            contexte += f"[{i}] {texte}\n\n"
            sources.append({"n": i, "source": src, "pertinence": round(sc, 2)})
        confiance = round(100 * sum(s["pertinence"] for s in sources) / len(sources))
        return contexte, sources, confiance
    # Hors-corpus local -> on tente le web ciblé
    res = tavily.search(query=question, include_domains=choisir_domaines(question), max_results=3)["results"]
    if res:
        contexte = "\n\n".join(r["content"] for r in res)
        sources = [{"n": i + 1, "source": r["url"], "pertinence": "web"} for i, r in enumerate(res)]
        return contexte, sources, 50
    return "", [], 0

def repondre_fiable(question, k=4):
    """Répond à une question avec citations + score de confiance."""
    contexte, sources, confiance = _contexte_numerote(question, k)
    if not sources:
        return {"reponse": "⚠️ Information absente de la bibliothèque. Ajoutez un document fiable.",
                "confiance": 0, "sources": [], "a_verifier": True}
    systeme = ("Tu es un professeur de santé au Cameroun. Réponds en français en t'appuyant UNIQUEMENT "
               "sur le contexte numéroté. Après CHAQUE affirmation, mets le numéro de la source [n]. "
               "Si une information manque, écris « (non documenté) ».\n\nContexte :\n" + contexte)
    return {"reponse": demander(systeme, question), "confiance": confiance,
            "sources": sources, "a_verifier": confiance < 60}

def generer_contenu_fiable(sujet, consigne, k=4):
    """Générique : produit un contenu sourcé (cours, activité, TP…) + confiance.
    Renvoie (contenu, sources, confiance, a_verifier)."""
    contexte, sources, confiance = _contexte_numerote(sujet, k)
    systeme = ("Tu es un enseignant de santé au Cameroun. Réponds en français en t'appuyant UNIQUEMENT "
               "sur le contexte numéroté. Après chaque affirmation issue du contexte, mets le numéro "
               "de la source [n]. Si une information manque, écris « (non documenté) ».\n\nContexte :\n" + contexte)
    return demander(systeme, consigne), sources, confiance, confiance < 60

def afficher_fiable(res):
    print(res["reponse"])
    etat = "⚠️ à vérifier par un humain" if res["a_verifier"] else "✅ fiable"
    print(f"\n🔎 Confiance : {res['confiance']} %  —  {etat}")
    for s in res["sources"]:
        print(f"   [{s['n']}] {s['source']}  (pertinence {s['pertinence']})")

print("✅ Brique fiabilité prête (recuperer_avec_scores, repondre_fiable, generer_contenu_fiable).")

### Essais de la fiabilité

Une question **couverte** par la bibliothèque (confiance élevée) puis une question **hors-corpus** (l'assistant doit refuser d'inventer).

In [ ]:
# 1) Question couverte
afficher_fiable(repondre_fiable("Comment prévenir le paludisme ?"))

print("\n" + "="*50 + "\n")

# 2) Question hors de la bibliothèque
afficher_fiable(repondre_fiable("Quel est le taux de TVA sur les panneaux solaires ?"))

## 4) Générateur de COURS (votre classe)

In [ ]:
"""
cours.py
Module de génération automatique de cours.
Nécessite :
    demander(systeme, question)
    chercher_contexte(sujet)
"""

import docx


class GenerateurCours:

    def __init__(self, demander, chercher_contexte,
                 pays="Cameroun",
                 langue="français"):

        self.demander = demander
        self.chercher_contexte = chercher_contexte
        self.pays = pays
        self.langue = langue

    # --------------------------------------------------

    def _systeme(self, sujet):

        contexte, sources, origine = self.chercher_contexte(sujet)

        systeme = f"""
Tu es un enseignant des sciences de la santé.

Pays : {self.pays}
Langue : {self.langue}

Utilise uniquement le contexte ci-dessous.

{contexte}
"""

        return systeme, sources

    # --------------------------------------------------

    def construire_plan(self, sujet, niveau, objectifs):

        systeme, _ = self._systeme(sujet)

        question = f"""
Construis un plan pédagogique de 6 parties.

Sujet : {sujet}

Niveau : {niveau}

Objectifs :
{objectifs}

Réponds avec uniquement les titres.
"""

        rep = self.demander(systeme, question)

        parties = []

        for ligne in rep.split("\n"):

            ligne = ligne.strip()

            ligne = ligne.lstrip("-•0123456789. ")

            if ligne:
                parties.append(ligne)

        return parties[:6]

    # --------------------------------------------------

    def introduction(self, sujet, niveau):

        systeme, _ = self._systeme(sujet)

        return self.demander(
            systeme,
            f"Rédige une introduction pédagogique sur {sujet} pour {niveau}."
        )

    # --------------------------------------------------

    def partie(self, sujet, titre, niveau):

        systeme, sources = self._systeme(titre)

        texte = self.demander(
            systeme,
            f"""
Sujet : {sujet}

Titre de la partie :

{titre}

Niveau : {niveau}

Rédige une partie structurée de 4 à 6 paragraphes.
"""
        )

        return texte, sources

    # --------------------------------------------------

    def conclusion(self, sujet):

        systeme, _ = self._systeme(sujet)

        return self.demander(
            systeme,
            f"Rédige une conclusion pédagogique sur {sujet}."
        )

    # --------------------------------------------------

    def generer(self,
                sujet,
                niveau,
                objectifs):

        plan = self.construire_plan(
            sujet,
            niveau,
            objectifs
        )

        contenu = {

            "titre": sujet,

            "niveau": niveau,

            "objectifs": objectifs,

            "plan": plan,

            "introduction": self.introduction(
                sujet,
                niveau
            ),

            "parties": [],

            "bibliographie": []
        }

        for p in plan:

            texte, sources = self.partie(
                sujet,
                p,
                niveau
            )

            contenu["parties"].append({

                "titre": p,

                "texte": texte

            })

            contenu["bibliographie"].extend(sources)

        contenu["conclusion"] = self.conclusion(
            sujet
        )

        contenu["bibliographie"] = sorted(
            set(contenu["bibliographie"])
        )

        return contenu

    # --------------------------------------------------

    def exporter_word(
            self,
            cours,
            fichier="Cours.docx"):

        doc = docx.Document()

        doc.add_heading(cours["titre"], 0)

        doc.add_heading("Niveau", 1)
        doc.add_paragraph(cours["niveau"])

        doc.add_heading("Objectifs", 1)
        doc.add_paragraph(cours["objectifs"])

        doc.add_heading("Introduction", 1)
        doc.add_paragraph(cours["introduction"])

        doc.add_heading("Développement", 1)

        for partie in cours["parties"]:

            doc.add_heading(
                partie["titre"],
                level=2
            )

            doc.add_paragraph(
                partie["texte"]
            )

        doc.add_heading(
            "Conclusion",
            1
        )

        doc.add_paragraph(
            cours["conclusion"]
        )

        doc.add_heading(
            "Bibliographie",
            1
        )

        for source in cours["bibliographie"]:

            doc.add_paragraph(
                source,
                style="List Bullet"
            )

        doc.save(fichier)

        return fichier

### Utiliser le générateur de cours

In [ ]:
gc = GenerateurCours(demander, chercher_contexte)
cours = gc.generer(
    sujet="La prévention du paludisme",
    niveau="infirmiers de 1re année",
    objectifs="Comprendre la transmission, les symptômes et la prévention.")
gc.exporter_word(cours, "Cours.docx")
print("📄 Cours.docx —", len(cours["parties"]), "parties")

## 5) Générateur d'ACTIVITÉS (votre classe)

Tous les types : définitions, QCM, vrai/faux, questions ouvertes, phrases à trous, association, tableau, schéma, étude de cas, TP, simulation, exposé, recherche documentaire, banque de questions.

In [ ]:
"""
activites.py - Module de génération d'activités pédagogiques
"""

import json
import docx
import pandas as pd

class GenerateurActivites:
    def __init__(self, demander, chercher_contexte, pays="Cameroun", langue="français"):
        self.demander = demander
        self.chercher_contexte = chercher_contexte
        self.pays = pays
        self.langue = langue

    def _systeme(self, sujet):
        contexte, sources, origine = self.chercher_contexte(sujet)
        systeme = f"""Tu es un enseignant en sciences de la santé.
Pays : {self.pays}
Langue : {self.langue}

Utilise uniquement le contexte suivant :

{contexte}
"""
        return systeme, sources

    def _generer(self, sujet, consigne, niveau="Licence", difficulte="Moyen", competences=None):
        competences = competences or []
        systeme, sources = self._systeme(sujet)
        question = f"""Sujet : {sujet}
Niveau : {niveau}
Difficulté : {difficulte}
Compétences : {", ".join(competences)}

{consigne}
"""
        texte = self.demander(systeme, question)
        return {
            "sujet": sujet,
            "niveau": niveau,
            "difficulte": difficulte,
            "competences": competences,
            "contenu": texte,
            "sources": sources,
        }

    def definitions(self, sujet, niveau="Licence"):
        return self._generer(sujet, "Donne 10 définitions avec exemples.", niveau)

    def qcm(self, sujet, niveau="Licence", nb=10, difficulte="Moyen", competences=None):
        return self._generer(
            sujet,
            f"Produis {nb} QCM avec compétence, question, A-D, bonne réponse et explication.",
            niveau,
            difficulte,
            competences,
        )

    def vrai_faux(self, sujet, niveau="Licence", nb=10):
        return self._generer(sujet, f"Produis {nb} affirmations Vrai/Faux avec justification.", niveau)

    def questions_ouvertes(self, sujet, niveau="Licence", nb=10):
        return self._generer(sujet, f"Produis {nb} questions ouvertes avec corrigé.", niveau)

    def phrases_trous(self, sujet, niveau="Licence", nb=10):
        return self._generer(sujet, f"Produis {nb} phrases à trous avec corrigé.", niveau)

    def association(self, sujet, niveau="Licence"):
        return self._generer(sujet, "Créer un exercice d'association avec corrigé.", niveau)

    def tableau(self, sujet, niveau="Licence"):
        return self._generer(sujet, "Créer un tableau à compléter puis son corrigé.", niveau)

    def schema(self, sujet, niveau="Licence"):
        return self._generer(sujet, "Décrire un schéma à compléter puis son corrigé.", niveau)

    def etude_cas(self, sujet, niveau="Licence"):
        return self._generer(sujet, "Créer une étude de cas complète avec corrigé.", niveau)

    def tp(self, sujet, niveau="Licence"):
        return self._generer(sujet, "Créer un TP complet avec objectifs, matériel, procédure, sécurité, grille et corrigé.", niveau)

    def simulation(self, sujet, niveau="Licence"):
        return self._generer(sujet, "Créer une simulation clinique complète.", niveau)

    def expose(self, sujet, niveau="Licence"):
        return self._generer(sujet, "Préparer un exposé structuré avec bibliographie.", niveau)

    def recherche_documentaire(self, sujet, niveau="Licence"):
        return self._generer(sujet, "Créer une activité de recherche documentaire.", niveau)

    def banque_questions(self, chapitres, niveau="Licence", nb=5):
        return [self.qcm(ch, niveau=niveau, nb=nb) for ch in chapitres]

    def exporter_word(self, activite, fichier="Activite.docx"):
        doc = docx.Document()
        doc.add_heading(activite["sujet"], 0)
        doc.add_paragraph(f'Niveau : {activite["niveau"]}')
        doc.add_paragraph(f'Difficulté : {activite["difficulte"]}')
        doc.add_heading("Contenu", 1)
        doc.add_paragraph(activite["contenu"])
        doc.add_heading("Sources", 1)
        for s in activite["sources"]:
            doc.add_paragraph(str(s), style="List Bullet")
        doc.save(fichier)
        return fichier

    def exporter_excel(self, banque, fichier="Banque.xlsx"):
        lignes = []
        for a in banque:
            lignes.append({
                "Sujet": a["sujet"],
                "Niveau": a["niveau"],
                "Difficulté": a["difficulte"],
                "Compétences": ", ".join(a["competences"]),
                "Contenu": a["contenu"],
            })
        pd.DataFrame(lignes).to_excel(fichier, index=False)
        return fichier

    def exporter_json(self, activite, fichier="Activite.json"):
        with open(fichier, "w", encoding="utf-8") as f:
            json.dump(activite, f, ensure_ascii=False, indent=4)
        return fichier

### Utiliser le générateur d'activités

In [ ]:
ga = GenerateurActivites(demander, chercher_contexte)

# Un QCM
qcm = ga.qcm("la prévention du paludisme", niveau="Licence", nb=3)
print(qcm["contenu"][:500], "...")
ga.exporter_word(qcm, "Activite_QCM.docx")

# Une banque de questions -> Excel
banque = ga.banque_questions(["paludisme", "vaccination"], nb=3)
ga.exporter_excel(banque, "Banque.xlsx")
print("📊 Banque.xlsx")

## 6) ÉVALUATION / référentiel (votre classe)

In [ ]:
"""
evaluation.py
Module d'évaluation pédagogique.
Dépendances :
    demander(systeme, question)
"""

import docx
import pandas as pd


class EvaluateurCours:

    def __init__(self, demander):
        self.demander = demander

    def evaluer_competence(self, cours_texte, competence):
        systeme = """
Tu es un expert en pédagogie.

Réponds exactement sous la forme :

OUI|Justification
ou
NON|Justification

Ne réponds rien d'autre.
"""

        question = f"""
Compétence :
{competence}

Cours :
{cours_texte}
"""

        try:
            rep = self.demander(systeme, question).strip()
        except Exception:
            return {
                "competence": competence,
                "couverte": False,
                "justification": "Erreur lors de l'évaluation.",
                "recommandation": "Réévaluer cette compétence."
            }

        ok = rep.upper().startswith("OUI")

        justification = rep.split("|", 1)[1].strip() if "|" in rep else rep

        recommandation = "" if ok else "Ajouter une section couvrant cette compétence."

        return {
            "competence": competence,
            "couverte": ok,
            "justification": justification,
            "recommandation": recommandation
        }

    def matrice(self, cours_texte, referentiel):

        lignes = []

        for comp in referentiel:
            lignes.append(self.evaluer_competence(cours_texte, comp))

        total = len(lignes)
        couvertes = sum(1 for l in lignes if l["couverte"])
        taux = round((couvertes / total) * 100, 1) if total else 0

        df = pd.DataFrame({
            "Compétence": [l["competence"] for l in lignes],
            "Couverte": ["✅" if l["couverte"] else "❌" for l in lignes],
            "Justification": [l["justification"] for l in lignes],
            "Recommandation": [l["recommandation"] for l in lignes],
        })

        return df, taux

    def exporter_excel(self, df, fichier="Matrice_Couverture.xlsx"):
        df.to_excel(fichier, index=False)
        return fichier

    def exporter_word(self, df, taux, fichier="Matrice_Couverture.docx"):
        doc = docx.Document()
        doc.add_heading("Matrice de couverture des compétences", 0)
        doc.add_paragraph(f"Taux de couverture : {taux}%")

        table = doc.add_table(rows=1, cols=4)
        table.style = "Table Grid"

        headers = ["Compétence", "Couverte", "Justification", "Recommandation"]

        for i, h in enumerate(headers):
            table.rows[0].cells[i].text = h

        for _, row in df.iterrows():
            cells = table.add_row().cells
            cells[0].text = str(row["Compétence"])
            cells[1].text = str(row["Couverte"])
            cells[2].text = str(row["Justification"])
            cells[3].text = str(row["Recommandation"])

        doc.save(fichier)
        return fichier

    def rapport(self, df, taux):
        total = len(df)
        couvertes = (df["Couverte"] == "✅").sum()
        manquantes = total - couvertes

        return {
            "total_competences": total,
            "competences_couvertes": int(couvertes),
            "competences_manquantes": int(manquantes),
            "taux_couverture": taux
        }

### Vérifier la couverture d'un référentiel

In [ ]:
ev = EvaluateurCours(demander)

REFERENTIEL = [
    "Expliquer la transmission du paludisme",
    "Reconnaître les symptômes",
    "Décrire les méthodes de prévention",
    "Expliquer le traitement de première intention",
]
cours_texte = "Le paludisme se transmet par le moustique. Symptômes : fièvre, frissons. Prévention : moustiquaire."

df, taux = ev.matrice(cours_texte, REFERENTIEL)
print("Taux de couverture :", taux, "%")
print(df)
ev.exporter_word(df, taux, "Matrice_Couverture.docx")
ev.exporter_excel(df, "Matrice_Couverture.xlsx")

## 7) SUPPORTS : PowerPoint + fiche de révision

In [ ]:
from pptx import Presentation

def points_cles(texte, n=4):
    rep = demander(f"Résume en {n} puces courtes, une par ligne.", texte)
    return [l.strip(" -*•\t").strip() for l in rep.split("\n") if l.strip()][:n]

def generer_powerpoint(sujet, fichier="Support.pptx"):
    contexte, _, _ = chercher_contexte(sujet)
    prs = Presentation()
    d = prs.slides.add_slide(prs.slide_layouts[0]); d.shapes.title.text = sujet; d.placeholders[1].text = "Support de cours"
    d = prs.slides.add_slide(prs.slide_layouts[1]); d.shapes.title.text = "Points clés"
    pts = points_cles(contexte or sujet, 5); cadre = d.placeholders[1].text_frame; cadre.text = pts[0] if pts else ""
    for p in pts[1:]:
        cadre.add_paragraph().text = p
    prs.save(fichier); print("📊", fichier)

def generer_fiche_revision(sujet, fichier="Fiche_revision.docx"):
    contexte, _, _ = chercher_contexte(sujet)
    resume = demander("Rédige un résumé clair de 5 lignes.", contexte or sujet)
    cles = points_cles(contexte or sujet, 5)
    doc = docx.Document(); doc.add_heading(f"Fiche de révision — {sujet}", 0)
    doc.add_heading("Résumé", 1); doc.add_paragraph(resume)
    doc.add_heading("Points clés", 1)
    for c in cles:
        doc.add_paragraph(c, style="List Bullet")
    doc.save(fichier); print("📝", fichier)

generer_powerpoint("la prévention du paludisme")
generer_fiche_revision("la prévention du paludisme")

## ✅ Récapitulatif

- **Socle unique** : `demander`, bibliothèque (PDF/Word/txt/CSV/Excel, multi-domaines, mémoire d'index), `chercher_contexte`, `ask`.
- **🛡️ Fiabilité** : `repondre_fiable` (citations + confiance) et `generer_contenu_fiable` (contenu sourcé + confiance) — le **même moteur** que l'application.
- **3 classes** : `GenerateurCours`, `GenerateurActivites`, `EvaluateurCours`.
- **Supports** : PowerPoint + fiche de révision.

L'application `app.py` réutilise ces mêmes fonctions dans une interface avec profils et onglets : onglet **Question** (citations + barre de confiance), **Cours** (confiance par partie + « à relire en priorité »), **Activités / TP / Banque / Supports** (score de confiance et sources partout).